# 🥇 TASK 4 — Gold Layer & Databricks SQL
**Notebook:** `gold_layer_databricks_sql`  
**Depends on:** `silver_layer_transformation` (Task 3 must be run first)  
**Steps covered:**
- Step 20 — View: gold.dept_clinical_summary (total visits, avg duration, avg liability, recovery rate)
- Step 21 — View: gold.monthly_revenue (total billed, total insurance covered per month)
- Step 22 — SQL Dashboard simulation (bar chart + line chart using display())
- Step 23 — Alert simulation: departments with > 10 emergency visits in a single day

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 1 — Imports & Spark session
# ─────────────────────────────────────────────────────────────
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

print(f"✅ Spark {spark.version} ready")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 2 — Catalog / schema / table references
# ─────────────────────────────────────────────────────────────
CATALOG = "databricks_project_tsv-752"   # ← your Unity Catalog name
SCHEMA  = "healthcare"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

# ── Silver source table (written in Task 3) ───────────────────
SILVER_VISITS = f"`{CATALOG}`.`{SCHEMA}`.silver_patient_visits"

# ── Gold view names ───────────────────────────────────────────
GOLD_DEPT_SUMMARY   = f"`{CATALOG}`.`{SCHEMA}`.gold_dept_clinical_summary"
GOLD_MONTHLY_REV    = f"`{CATALOG}`.`{SCHEMA}`.gold_monthly_revenue"

print(f"✅ Context: {CATALOG}.{SCHEMA}")
print(f"   Source   → {SILVER_VISITS}")
print(f"   Gold 1   → {GOLD_DEPT_SUMMARY}")
print(f"   Gold 2   → {GOLD_MONTHLY_REV}")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 3 — Validate Silver source is available
# ─────────────────────────────────────────────────────────────
cnt = spark.table(SILVER_VISITS).count()
print(f"📊 silver.patient_visits row count: {cnt:,}")

print("\n📋 Columns available for Gold aggregation:")
for field in spark.table(SILVER_VISITS).schema.fields:
    print(f"   {field.name:30s} {str(field.dataType):20s}")

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 4 — STEP 20
# Create view: gold.dept_clinical_summary
#
# Aggregates per department:
#   total_visits         — count of all visits
#   avg_duration_days    — average length of stay
#   avg_patient_liability— average out-of-pocket cost
#   recovery_rate        — % of visits where discharge_status = 'recovered'
#
# Using CREATE OR REPLACE VIEW so the notebook is idempotent
# ─────────────────────────────────────────────────────────────

spark.sql(f"""
    CREATE OR REPLACE VIEW {GOLD_DEPT_SUMMARY} AS
    SELECT
        department,

        -- Volume KPI
        COUNT(visit_id)                                          AS total_visits,

        -- Clinical KPIs
        ROUND(AVG(duration_days), 2)                            AS avg_duration_days,

        -- Financial KPI
        ROUND(AVG(patient_liability), 2)                        AS avg_patient_liability,

        -- Recovery rate: recovered visits / total visits * 100
        ROUND(
            SUM(CASE WHEN discharge_status = 'recovered' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(visit_id),
            2
        )                                                       AS recovery_rate_pct,

        -- Supporting detail for transparency
        SUM(CASE WHEN discharge_status = 'recovered' THEN 1 ELSE 0 END) AS recovered_count,
        SUM(CASE WHEN discharge_status = 'deceased'  THEN 1 ELSE 0 END) AS deceased_count,
        SUM(CASE WHEN discharge_status = 'referred'  THEN 1 ELSE 0 END) AS referred_count

    FROM {SILVER_VISITS}
    GROUP BY department
    ORDER BY total_visits DESC
""")

print(f"✅ View created: {GOLD_DEPT_SUMMARY}")
print("\n📊 dept_clinical_summary:")
display(spark.sql(f"SELECT * FROM {GOLD_DEPT_SUMMARY}"))

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 5 — STEP 21
# Create view: gold.monthly_revenue
#
# Aggregates per calendar month (yyyy-MM):
#   total_billed_amount  — sum of total_bill across all visits
#   total_insurance_covered — sum of insurance_covered
#   total_patient_liability — sum of patient_liability (net revenue)
#   visit_count         — number of visits in that month
# ─────────────────────────────────────────────────────────────

spark.sql(f"""
    CREATE OR REPLACE VIEW {GOLD_MONTHLY_REV} AS
    SELECT
        -- Extract year-month for grouping (e.g. 2024-04)
        DATE_FORMAT(visit_date, 'yyyy-MM')              AS revenue_month,
        YEAR(visit_date)                                AS revenue_year,
        MONTH(visit_date)                               AS revenue_month_num,

        COUNT(visit_id)                                 AS visit_count,

        -- Revenue KPIs (rounded to 2 decimal places)
        ROUND(SUM(total_bill), 2)                       AS total_billed_amount,
        ROUND(SUM(insurance_covered), 2)                AS total_insurance_covered,
        ROUND(SUM(patient_liability), 2)                AS total_patient_liability,

        -- Insurance coverage ratio
        ROUND(
            SUM(insurance_covered) * 100.0 / NULLIF(SUM(total_bill), 0),
            2
        )                                               AS insurance_coverage_pct

    FROM {SILVER_VISITS}
    WHERE visit_date IS NOT NULL
    GROUP BY
        DATE_FORMAT(visit_date, 'yyyy-MM'),
        YEAR(visit_date),
        MONTH(visit_date)
    ORDER BY revenue_month
""")

print(f"✅ View created: {GOLD_MONTHLY_REV}")
print("\n📊 monthly_revenue:")
display(spark.sql(f"SELECT * FROM {GOLD_MONTHLY_REV}"))

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — STEP 22  (Dashboard Simulation — Part A)
# BAR CHART: Total Visits by Department
#
# In a real Databricks SQL Dashboard:
#   1. Open Databricks SQL → Dashboards → New Dashboard
#   2. Add a Visualization widget → Query: SELECT department, total_visits
#      FROM gold_dept_clinical_summary ORDER BY total_visits DESC
#   3. Set chart type = Bar, X-axis = department, Y-axis = total_visits
#
# Below we simulate the same using display() with plot options.
# In Databricks notebooks, clicking the chart icon on a display()
# result lets you switch to Bar / Line view interactively.
# ─────────────────────────────────────────────────────────────

print("📊 DASHBOARD CHART 1 — Visits by Department (Bar Chart)")
print("   ℹ️  In Databricks: click the chart icon below → select Bar → set X=department, Y=total_visits")
print()

df_bar = spark.sql(f"""
    SELECT
        department,
        total_visits,
        avg_duration_days,
        recovery_rate_pct
    FROM {GOLD_DEPT_SUMMARY}
    ORDER BY total_visits DESC
""")

# display() renders a chart-ready table in Databricks notebooks
display(df_bar)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 7 — STEP 22  (Dashboard Simulation — Part B)
# LINE CHART: Monthly Revenue vs Insurance Coverage
#
# In a real Databricks SQL Dashboard:
#   1. Add a second Visualization widget
#   2. Query: SELECT revenue_month, total_billed_amount, total_insurance_covered
#      FROM gold_monthly_revenue ORDER BY revenue_month
#   3. Chart type = Line, X-axis = revenue_month,
#      Y-axis series: total_billed_amount AND total_insurance_covered
#
# Below: display() output → switch to Line chart in notebook UI.
# ─────────────────────────────────────────────────────────────

print("📊 DASHBOARD CHART 2 — Monthly Revenue vs Insurance (Line Chart)")
print("   ℹ️  In Databricks: click chart icon → Line → X=revenue_month, Y=[total_billed_amount, total_insurance_covered]")
print()

df_line = spark.sql(f"""
    SELECT
        revenue_month,
        total_billed_amount,
        total_insurance_covered,
        total_patient_liability,
        insurance_coverage_pct
    FROM {GOLD_MONTHLY_REV}
    ORDER BY revenue_month
""")

display(df_line)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 8 — STEP 22  (Dashboard Setup Instructions — SQL)
# Real Databricks SQL Dashboard configuration steps as comments
# The SQL queries below are exactly what you paste into the
# Databricks SQL editor to build each dashboard widget.
# ─────────────────────────────────────────────────────────────

# ── SQL QUERY 1: Bar chart widget ────────────────────────────
BAR_CHART_SQL = f"""
-- Dashboard Widget 1: Visits by Department (Bar Chart)
-- Paste this into Databricks SQL → New Query → Save → Add to Dashboard
SELECT
    department,
    total_visits,
    avg_duration_days,
    recovery_rate_pct
FROM {GOLD_DEPT_SUMMARY}
ORDER BY total_visits DESC;
"""

# ── SQL QUERY 2: Line chart widget ───────────────────────────
LINE_CHART_SQL = f"""
-- Dashboard Widget 2: Monthly Revenue vs Insurance (Line Chart)
-- Paste this into Databricks SQL → New Query → Save → Add to Dashboard
SELECT
    revenue_month,
    total_billed_amount,
    total_insurance_covered,
    total_patient_liability,
    insurance_coverage_pct
FROM {GOLD_MONTHLY_REV}
ORDER BY revenue_month;
"""

print("=" * 70)
print("📋 DATABRICKS SQL DASHBOARD SETUP INSTRUCTIONS")
print("=" * 70)
print("""
Step-by-step to build the visual dashboard in Databricks SQL:

  1. Navigate to:  Databricks Workspace → SQL → Dashboards → [+ New Dashboard]
     Name it: 'Healthcare Clinical & Revenue Dashboard'

  2. Widget 1 — Bar Chart (Visits by Department):
     a. Click [Add visualization] → [New query]
     b. Paste the BAR_CHART_SQL query (printed below)
     c. Run query → Click [+ Add visualization]
     d. Type: Bar Chart  |  X: department  |  Y: total_visits
     e. Save visualization → Add to dashboard

  3. Widget 2 — Line Chart (Monthly Revenue):
     a. Click [Add visualization] → [New query]
     b. Paste the LINE_CHART_SQL query (printed below)
     c. Run query → Click [+ Add visualization]
     d. Type: Line Chart  |  X: revenue_month
        Y series 1: total_billed_amount  (label: Total Billed)
        Y series 2: total_insurance_covered  (label: Insurance Covered)
     e. Save visualization → Add to dashboard

  4. Click [Publish] to share the dashboard with your team.
""")
print("──── BAR CHART SQL ────")
print(BAR_CHART_SQL)
print("──── LINE CHART SQL ────")
print(LINE_CHART_SQL)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 9 — STEP 23
# Alert: departments with > 10 emergency visits in a single day
#
# In Databricks SQL Alerts:
#   1. Save the query below as 'Emergency Visit Alert Query'
#   2. Go to Alerts → New Alert → Select that query
#   3. Condition: value of `emergency_visit_count` > 10
#   4. Schedule: Every 1 hour (or as needed)
#   5. Notification: Email / Slack webhook
#
# Below we execute the same logic in PySpark and flag breaches.
# ─────────────────────────────────────────────────────────────

# ── PySpark implementation of the alert query ─────────────────
df_emergency_alert = (
    spark.table(SILVER_VISITS)
    .filter(F.col("visit_type") == "emergency")
    .groupBy("department", "visit_date")
    .agg(
        F.count("visit_id").alias("emergency_visit_count")
    )
    .filter(F.col("emergency_visit_count") > 10)   # Alert threshold
    .orderBy(F.col("emergency_visit_count").desc())
)

alert_count = df_emergency_alert.count()

print("=" * 70)
print("🚨 ALERT: Departments with > 10 Emergency Visits in a Single Day")
print("=" * 70)

if alert_count > 0:
    print(f"  ⚠️  ALERT TRIGGERED — {alert_count} department-day combination(s) exceeded threshold!")
    display(df_emergency_alert)
else:
    print("  ✅ No alert — no department exceeded 10 emergency visits on any single day.")
    # Show top emergency days for transparency
    print("\n  📊 Top 10 emergency days (for reference):")
    display(
        spark.table(SILVER_VISITS)
        .filter(F.col("visit_type") == "emergency")
        .groupBy("department", "visit_date")
        .agg(F.count("visit_id").alias("emergency_visit_count"))
        .orderBy(F.col("emergency_visit_count").desc())
        .limit(10)
    )

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 10 — STEP 23 (continued)
# Print the exact SQL to paste into Databricks SQL Alerts UI
# ─────────────────────────────────────────────────────────────

ALERT_SQL = f"""
-- Alert Query: Emergency Visit Threshold
-- Databricks SQL Alerts → New Alert → Paste this query
-- Condition: emergency_visit_count > 10
-- Schedule: Every 1 hour
-- Notification: your-team@hospital.com

SELECT
    department,
    visit_date,
    COUNT(visit_id) AS emergency_visit_count
FROM {SILVER_VISITS}
WHERE visit_type = 'emergency'
  AND visit_date = CURRENT_DATE()          -- restrict to today for real-time alerting
GROUP BY department, visit_date
HAVING COUNT(visit_id) > 10
ORDER BY emergency_visit_count DESC;
"""

print("=" * 70)
print("📋 DATABRICKS SQL ALERT SETUP INSTRUCTIONS")
print("=" * 70)
print("""
  1. Go to: Databricks SQL → Alerts → [+ New Alert]
  2. Click 'New query' and paste the SQL printed below.
  3. Name the query: 'Emergency Visit Alert Query'
  4. Run once to confirm results → Save query.
  5. Back in Alert setup:
       Trigger when: emergency_visit_count > 10
       Schedule    : Every 1 hour
       Notification: Email → your-team@hospital.com
                     (or Slack webhook / PagerDuty)
  6. Click [Create Alert] → [Enable Alert]
""")
print("──── ALERT SQL ────")
print(ALERT_SQL)

In [0]:
# ─────────────────────────────────────────────────────────────
# CELL 11 — Final Gold Layer Validation
# Confirm both views are queryable and return correct columns
# ─────────────────────────────────────────────────────────────

print("=" * 70)
print("🏆 GOLD LAYER — SUMMARY")
print("=" * 70)

gold_views = {
    "gold_dept_clinical_summary" : GOLD_DEPT_SUMMARY,
    "gold_monthly_revenue"       : GOLD_MONTHLY_REV,
}
for label, view in gold_views.items():
    cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {view}").collect()[0]["c"]
    print(f"  ✅ {label:35s}  {cnt:>4} rows")

print("\n" + "─" * 70)
print("📋 dept_clinical_summary — final preview:")
display(spark.sql(f"SELECT * FROM {GOLD_DEPT_SUMMARY}"))

print("\n📋 monthly_revenue — final preview:")
display(spark.sql(f"SELECT * FROM {GOLD_MONTHLY_REV}"))